In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py
import numpy as np

hdf5_path = '../data/EPIC_audio.hdf5' 


print(f"Tentativo di apertura del file: {hdf5_path}...\n")

with h5py.File(hdf5_path, 'r') as f:
    
    keys = list(f.keys())
    print(f"File aperto con successo.")
    print(f"Numero totale di chiavi (elementi) nel file: {len(keys)}")
    
    print(f"Prime 5 chiavi: {keys[:5]}\n")
    
    if len(keys) > 0:
        first_key = keys[0]
        audio_data = f[first_key]
        
        print(f"--- Analisi del primo elemento: '{first_key}' ---")
        print(f"Classe dell'oggetto HDF5: {type(audio_data)}")
        
        print(f"Forma (Shape) dei dati: {audio_data.shape}") 
        
        print(f"Tipo di dato (dtype): {audio_data.dtype}\n")
        
        primi_10_campioni = audio_data[:10]
        print(f"Primi 10 valori grezzi dell'audio:\n{primi_10_campioni}")

Tentativo di apertura del file: ../data/EPIC_audio.hdf5...

File aperto con successo.
Numero totale di chiavi (elementi) nel file: 712
Prime 5 chiavi: ['P01_01', 'P01_02', 'P01_03', 'P01_04', 'P01_05']

--- Analisi del primo elemento: 'P01_01' ---
Classe dell'oggetto HDF5: <class 'h5py._hl.dataset.Dataset'>
Forma (Shape) dei dati: (39651328,)
Tipo di dato (dtype): float32

Primi 10 valori grezzi dell'audio:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [3]:
import sys

sys.path.append('../src') 
from datasets.epic_sounds import EPICSoundsDataset

dataset = EPICSoundsDataset(
    annotations_file='../data/epic-sounds-annotations/EPIC_Sounds_train.csv',
    hdf5_path='../data/EPIC_audio.hdf5'
)

print(f"Numero totale di clip audio pronte: {len(dataset)}")

spettrogramma, etichetta = dataset[0]

print(f"Etichetta (Classe ID): {etichetta}")
print(f"Shape dell'immagine dello Spettrogramma: {spettrogramma.shape}")

Numero totale di clip audio pronte: 60055
Etichetta (Classe ID): 6
Shape dell'immagine dello Spettrogramma: torch.Size([1, 128, 1024])


In [4]:
from torch.utils.data import DataLoader

# Inizializziamo il DataLoader passando la nostra istanza di EPICSoundsDataset
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Iteriamo sul DataLoader per estrarre il primo batch
batch_spectrograms, batch_labels = next(iter(train_loader))

print(f"Shape del batch di spettrogrammi: {batch_spectrograms.shape}")
print(f"Shape del batch di etichette: {batch_labels.shape}")

Shape del batch di spettrogrammi: torch.Size([32, 1, 128, 1024])
Shape del batch di etichette: torch.Size([32])


In [ ]:
import sys
sys.path.append('../src')
import torch
from datasets.epic_sounds import EPICSoundsDataset
from torch.utils.data import DataLoader
from models.ast import EPICASTBaseline

# 1. Carichiamo il VERO dataset 
# (Grazie alla tua modifica, ora estrarrà in automatico 1024 frame)
dataset = EPICSoundsDataset(
    annotations_file='../data/epic-sounds-annotations/EPIC_Sounds_train.csv',
    hdf5_path='../data/EPIC_audio.hdf5'
)

# 2. Estraiamo un batch reale dal DataLoader
train_loader = DataLoader(dataset, batch_size=4, shuffle=True)
real_input_batch, labels = next(iter(train_loader))

print(f"Shape del batch reale: {real_input_batch.shape}") 
# Se vedi [32, 1, 128, 1024], il tuo fix ha funzionato alla perfezione!

# 3. Testiamo il modello
print("Inizializzazione AST...")
model = EPICASTBaseline(num_classes=44)

print("Esecuzione del forward pass...")
logits = model(real_input_batch)

print(f"Shape dei logits di output: {logits.shape}")

Shape del batch reale: torch.Size([32, 1, 128, 1024])
Inizializzazione AST...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Esecuzione del forward pass...
Shape dei logits di output: torch.Size([32, 44])


In [6]:
import pandas as pd

# Leggiamo il file delle annotazioni
df_train = pd.read_csv('../data/epic-sounds-annotations/EPIC_Sounds_train.csv')

# Calcoliamo il numero di classi uniche. 
# Assumiamo che gli ID partano da 0 fino a N-1.
NUM_CLASSES = df_train['class_id'].nunique()
print(f"Numero esatto di classi nel dataset EPIC-Sounds: {NUM_CLASSES}")

Numero esatto di classi nel dataset EPIC-Sounds: 44


In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. Istanziamo il modello con il numero corretto di classi
print(f"Inizializzazione AST con {NUM_CLASSES} classi...")
model = EPICASTBaseline(num_classes=NUM_CLASSES)

# Trasferiamo il modello sul dispositivo di calcolo ottimale.
# Se hai un Mac con Apple Silicon (M1/M2/M3), PyTorch usa 'mps' (Metal Performance Shaders) come GPU.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Utilizzo del dispositivo hardware: {device}")
model.to(device)

# 2. Definiamo Loss e Optimizer
criterion = nn.CrossEntropyLoss()
# Passiamo all'ottimizzatore solo i parametri che richiedono l'aggiornamento dei gradienti
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 3. Impostiamo il modello in modalità "Training" (attiva Dropout, BatchNorm, etc.)
model.train()

print("\n--- INIZIO TRAINING LOOP (Test su 2 Batch) ---")

# Iteriamo sul DataLoader
for batch_idx, (inputs, labels) in enumerate(train_loader):
    
    # A. Spostiamo i tensori sulla GPU (o CPU)
    inputs = inputs.to(device)
    labels = labels.to(device)
    
    # B. Step 1: Azzeramento gradienti
    optimizer.zero_grad()
    
    # C. Step 2: Forward pass
    outputs = model(inputs)
    
    # D. Step 3: Calcolo della Loss
    loss = criterion(outputs, labels)
    
    # E. Step 4: Backpropagation (Calcolo dei gradienti)
    loss.backward()
    
    # F. Step 5: Aggiornamento dei pesi
    optimizer.step()
    
    # Stampa diagnostica
    print(f"Batch [{batch_idx+1}] completato | Loss scalare: {loss.item():.4f}")
    
    # Interrompiamo artificialmente il loop dopo 2 batch per il test
    if batch_idx == 1:
        print("Test del Training Loop concluso con successo. Nessun errore architetturale rilevato.")
        break

Inizializzazione AST con 44 classi...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Utilizzo del dispositivo hardware: mps

--- INIZIO TRAINING LOOP (Test su 2 Batch) ---


RuntimeError: MPS backend out of memory (MPS allocated: 17.24 GB, other allocations: 2.11 GB, max allowed: 20.13 GB). Tried to allocate 2.11 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

: 